# Python and C++ extension

## Importing library

In [1]:
import numpy as np
import argparse
import os.path
import scipy.sparse
import vtk
from pypolydim import polydim, gedim
from pypolydim.export_vtk_utilities import ExportVTKUtilities
from pypolydim.assembler_utilities import assembler_utilities

import sys
sys.path.insert(1, '../')
import other_utilities as other_ut

### Initialize

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('-order','--method-order',dest='method_order', default=3, type=int, help="Method order")
parser.add_argument('-method','--method-type',dest='method_type', default=0, type=int, help="Method type")
parser.add_argument('-test', '--test-id', dest='test_id', default=1, type=int, help="Test type")
parser.add_argument('-mesh', '--mesh-type', dest='mesh_type', default=0, type=int, help="Mesh type")
parser.add_argument('-tol1', '--tolerance-1-d', dest='tolerance1_d', default=1.0e-6, type=float, help="Geometric Tolerance 1D")
parser.add_argument('-tol2', '--tolerance-2-d', dest='tolerance2_d', default=1.0e-12, type=float, help="Geometric Tolerance 2D")
parser.add_argument('-area', '--mesh-max-relative-area', dest='max_relative_area', default=0.1, type=float, help="Mesh max relative area")
parser.add_argument('-export', '--export-path', dest='export_path', default='./Export', type=str, help="Export Path")
parser.add_argument('-import', '--import-path', dest='import_path', default='./', type=str, help="Mesh Import Path")
args = parser.parse_args([])

geometry_utilities_config = gedim.GeometryUtilitiesConfig()
geometry_utilities_config.tolerance1_d = args.tolerance1_d
geometry_utilities_config.tolerance2_d = args.tolerance2_d
geometry_utilities = gedim.GeometryUtilities(geometry_utilities_config)
mesh_utilities = gedim.MeshUtilities()
vtk_utilities = ExportVTKUtilities()

# Export folder
export_file_path = args.export_path
if not os.path.exists(export_file_path):
    os.makedirs(export_file_path)

# Mesh file path
export_mesh_path = args.export_path + "/Mesh"
if not os.path.exists(export_mesh_path):
    os.makedirs(export_mesh_path)

# Solution file path
export_solution_path = args.export_path + "/Solution"
if not os.path.exists(export_solution_path):
    os.makedirs(export_solution_path)

## Patch Test

Solving the following equation on square $\bar{\Omega} = [0, 1] \times [0, 1]$

$$
\begin{cases}
\nabla \cdot (a \nabla u) + b \cdot \nabla u + c u = f & \text{in } \Omega\\
a \nabla u \cdot n_1 = a \nabla u_{ex} \cdot n_1 & \text{in } \Gamma_{top}\\
a \nabla u \cdot n_2 = a \nabla u_{ex} \cdot n_2 & \text{in } \Gamma_{right}\\
u = u_{ex} & \text{in } ∂Ω∖ (\Gamma_{top} ∪ \Gamma_{right})
\end{cases}
$$

where $u_{ex} = (x + y + 0.5)^k$.

### Define Simulation Parameters

We define the square domain and the simulation parameters

In [ ]:
pde_domain = polydim.pde_tools.mesh.pde_mesh_utilities.PDE_Domain_2D()
pde_domain.vertices = np.array([[0.0, 1.0, 1.0, 0.0],
                                [0.0, 0.0, 1.0, 1.0],
                                [0.0, 0.0, 0.0, 0.0]])
pde_domain.shape_type = polydim.pde_tools.mesh.pde_mesh_utilities.PDE_Domain_2D.Domain_Shape_Types.parallelogram
pde_domain.area = 1.0

In [ ]:
mesh_type = polydim.pde_tools.mesh.pde_mesh_utilities.MeshGenerator_Types_2D(args.mesh_type)
method_type = polydim.pde_tools.local_space_pcc_2_d.MethodTypes(args.method_type)
method_order = args.method_order
mesh_size = float(args.max_relative_area)

### Create Mesh

We create the mesh using the domain definition contained in the `pde_domain` object and the `mesh_size` selected.

In [ ]:
mesh_data = gedim.MeshMatrices()
mesh = gedim.MeshMatricesDAO(mesh_data)

polydim.pde_tools.mesh.pde_mesh_utilities.create_mesh_2_d(geometry_utilities,
                                                          mesh_utilities,
                                                          mesh_type,
                                                          pde_domain,
                                                          mesh_size,
                                                          mesh)
mesh_geometric_data = polydim.pde_tools.mesh.pde_mesh_utilities.compute_mesh_2_d_geometry_data(geometry_utilities, mesh_utilities, mesh)

vtk_utilities.export_mesh(export_mesh_path, mesh)

To description of the domain borders are passed to the library for the vertices and the edges of the domain as integer values called `markers`.
Each `marker` identifies a different boundary condition.

__In this example__:

 `marker=2` identifies $Γ_{right}$, `marker=4` identifies $Γ_{top}$ and `marker=1` identifies the Dirichlet boundary.

In [ ]:
info_internal = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.none)
info_internal.marker = 0

info_dirichlet = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.strong)
info_dirichlet.marker = 1

info_neumann_top = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.weak)
info_neumann_top.marker = 4

info_neumann_right = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.weak)
info_neumann_right.marker = 2

info_neumann_none = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.none)
info_neumann_none.marker = 0

boundary_info = {
    0: info_internal,
    1: info_dirichlet,
    2: info_dirichlet,
    3: info_neumann_none,
    4: info_dirichlet,
    5: info_dirichlet,
    6: info_neumann_right,
    7: info_neumann_top,
    8: info_dirichlet
}

### Create Discrete Space FEM

The boundary condition types are passed to the library during the creation of the discrete space.
The types are the following:
* `BoundaryConditionType=1`: internal mesh point;
* `BoundaryConditionType=2`: strong boundary mesh point (Dirichlet in this example);
* `BoundaryConditionType=3`: weak boundary mesh point (Neumann in this example).
The array `BoundaryConditionType` describes for each `marker` the type of boundary condition associated.

__NB__: the array has dimension `num_markers+1`, as the first element is associated to non-usable `marker=0`.

__In this example__:

we have $3$ different markers, thus `BoundaryConditionsType` has size $3+1=4$. In particular `marker=1` has type `BoundaryConditionsType[1]=2` (Dirichlet), `marker=2` has type `BoundaryConditionsType[2]=3` and `marker=3` (Neumann) has type `BoundaryConditionsType[3]=3` (Neumann).

In [ ]:
mesh_connectivity_data = polydim.pde_tools.mesh.MeshMatricesDAO_mesh_connectivity_data(mesh)

trial_reference_element_data = polydim.pde_tools.local_space_pcc_2_d.create_reference_element(method_type, method_order)
test_reference_element_data = polydim.pde_tools.local_space_pcc_2_d.create_reference_element(method_type, method_order)

dof_manager = polydim.pde_tools.do_fs.DOFsManager()

trial_mesh_dofs_info = polydim.pde_tools.local_space_pcc_2_d.set_mesh_do_fs_info(trial_reference_element_data, mesh, boundary_info)
trial_dofs_data = dof_manager.create_do_fs_2_d(trial_mesh_dofs_info, mesh_connectivity_data)
test_mesh_dofs_info = polydim.pde_tools.local_space_pcc_2_d.set_mesh_do_fs_info(trial_reference_element_data, mesh, boundary_info)
test_dofs_data = dof_manager.create_do_fs_2_d(test_mesh_dofs_info, mesh_connectivity_data)

### Assemble linear system

In [ ]:
a = 3.0
b = np.array([0.1, 0.02, 0.03])
c = 2.0

def exact_solution_function(x, y, z):
    polynomial = x + y + 0.5
    result = 1.0
    max_order = method_order
    for i in range(max_order):
        result *= polynomial
    return result           

def exact_gradient_solution_function(x, y, z):
    polynomial = x + y + 0.5
    result = method_order
    max_order = method_order - 1
    for i in range(max_order):
        result *= polynomial
    return np.array([result, result, 0.0])

def exact_laplacian_solution_function(x, y, z):
    polynomial = x + y + 0.5
    result = 2.0 * method_order * (method_order - 1);
    max_order = method_order - 2
    for i in range(max_order):
        result *= polynomial
    return result

In [ ]:
def source_term_function(x, y, z):
    u_lap = exact_laplacian_solution_function(x, y, z)
    u_grad = exact_gradient_solution_function(x, y, z)
    u = exact_solution_function(x, y, z)

    return -a * u_lap + (b[0] * u_grad[0] + b[1] * u_grad[1] + b[2] * u_grad[2]) + c * u;

source_term = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       test_dofs_data,
                                                                       trial_reference_element_data,
                                                                       test_reference_element_data,
                                                                       source_term_function)                                                                       

In [ ]:
def strong_solution_function(marker, x, y, z):  
    return exact_solution_function(x, y, z)

u_D = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_strong_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_mesh_dofs_info,
                                                                       trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                       strong_solution_function)

In [ ]:
def weak_term_function(marker, x, y, z):
    u_grad = exact_gradient_solution_function(x, y, z)
    match marker:
        case 2:
            return a * u_grad[0] 
        case 4:
            return a * u_grad[1] 
        case _:
            raise ValueError("not valid marker")


weak_term = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_weak_term(geometry_utilities,
                                                                             mesh,
                                                                             mesh_geometric_data,
                                                                             trial_mesh_dofs_info,
                                                                             test_dofs_data,
                                                                             trial_reference_element_data,
                                                                             test_reference_element_data,
                                                                             weak_term_function)

In [ ]:
def diffusion_term_function(x, y, z):  
    return a
def advection_term_function(x, y, z):  
    return b
def reaction_term_function(x, y, z):  
    return c

diffusion = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_diffusion_operator(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_dofs_data,
                                                                                             test_dofs_data,
                                                                       trial_reference_element_data,
                                                                                             test_reference_element_data,
                                                                       diffusion_term_function)


A1 = other_ut.make_np_sparse(diffusion.operator_dofs)
A1_D = other_ut.make_np_sparse(diffusion.operator_strong)

advection = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_advection_operator(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_dofs_data,
                                                                                             test_dofs_data,
                                                                       trial_reference_element_data,
                                                                                             test_reference_element_data,
                                                                       advection_term_function)


A2 = other_ut.make_np_sparse(advection.operator_dofs)
A2_D = other_ut.make_np_sparse(advection.operator_strong)

reaction = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_reaction_operator(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_dofs_data,
                                                                                             test_dofs_data,
                                                                       trial_reference_element_data,
                                                                                             test_reference_element_data,
                                                                       reaction_term_function)


A3 = other_ut.make_np_sparse(reaction.operator_dofs)
A3_D = other_ut.make_np_sparse(reaction.operator_strong)

A = A1 + A2 + A3
A_D = A1_D + A2_D + A3_D

### Solve linear system

In [ ]:
rhs = source_term + weak_term - A_D @ u_D
u = scipy.sparse.linalg.spsolve(A, rhs)

### Compute errors

In [ ]:
u_ext = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_exact_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                             trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                       exact_solution_function)

In [ ]:
u_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                       trial_dofs_data,
                                                                                         u,
                                                                                         u_D,
                                                                       exact_solution_function)

In [ ]:
error_L2 = polydim.pde_tools.assembler_utilities.pcc_2_d.compute_error_l2(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                          u,
                                                                          u_D,
                                                                       exact_solution_function)

error_H1 = polydim.pde_tools.assembler_utilities.pcc_2_d.compute_error_h1(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                          u,
                                                                          u_D,
                                                                       exact_gradient_solution_function)


print("dofs", "errorL2", "errorH1")
print(trial_dofs_data.number_do_fs, '{:.2e}'.format(error_L2.error_l2 / error_L2.numeric_norm_l2), '{:.2e}'.format(error_H1.error_h1 / error_H1.numeric_norm_h1))

In [ ]:
vtk_utilities.export_solution_2(export_solution_path + '/Solution.vtu',
                                mesh, 
                                u_on_cell0Ds.numeric_solution,
                                u_on_cell0Ds.exact_solution,
                                error_L2.cell2_ds_error_l2,
                               error_H1.cell2_ds_error_h1)